In [15]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import netCDF4 as n
import glob
import xarray as xr
import seaborn as sns
import matplotlib.lines as mlines
import pandas as pd
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.stats import linregress
from collections import Counter
sns.set_context("paper")
sns.set_style("whitegrid")
computed_name = 'ComputedScalarsHelene'
mnt_pth = '/Volumes/FMac/computing/data/antarctica/' #mount path
pth_ismip6hack= mnt_pth + 'ismip6/hackathon/ComputedScalars/' #write folder
pth_ismip6 =mnt_pth + 'ismip6/hackathon/' #read folder
pth_helene =mnt_pth + 'ismip6/hackathon/ComputedScalarsHelene/'
pth_table = '/Users/fmcc0003/issm/projects/ISMIP6Hackathon/GroupBMBtodSLR/data_analysis/tables/'

In [16]:
# choose variable,factor and path
factor = -1* 365.25 * 24 * 3600/900

var_area = 'iareafl'
var_bmb = 'shelfmelt'

var_dslc = 'dslc_anom'
factor = -1

fpath = f'/Volumes/FMac/computing/data/antarctica/ismip6/hackathon/figures/recalc_{var_dslc}_dslcs_dils/'
fname = f'cumulative_{var_dslc}_'

In [17]:
expnames = ['expAE02', 'expAE03', 'expAE04', 'expAE05']
removeFileID = ['IMAU_UFEMISM1', 'IMAU_UFEMISM2', 'IMAU_UFEMISM3', 'IMAU_UFEMISM4',
               'DOE_MALI_4km', 'DOE_MALI_8km_Ant95', 'DOE_MALI_8km_AntMean'] # specify models to remove
 
metadata_file = 'Metadata.txt'
def get_model(string,exp):
    return string.split('/')[-1].split('AIS_')[-1].split('_'+str(exp))[0]

# Generate loop_info DataFrame
models=[]
paths =[]
exps =[]
for exp in expnames:
    lst = glob.glob(pth_ismip6hack+f'{exp}/{var_dslc}/*.nc')
    lst.sort()

    for name in lst:
        model = get_model(name,exp)
        if model in removeFileID:
            continue
        else:
            models.append(model)
            paths.append(name)
            exps.append(exp)
df = pd.DataFrame({ 'Experiment': exps,'Model': models, 'Path': paths}) 
for i in range(1,19):
    df['sector_'+str(i)]=0
df['AIS']=0
df_org = df.copy()

metadata_file = '/Users/fmcc0003/issm/projects/ISMIP6Hackathon/GroupTFtoOcean/data_analysis/Metadata_icefront.txt'
df_info = pd.read_csv(metadata_file)
df_dils = pd.read_csv('./tables/linear_dslc_sensitivity_from_BMB_max.csv')
df_dils_n = pd.read_csv('./tables/linear_dslc_sensitivity_from_BMB_max_intercept.csv')

In [18]:
dic_area_of_interest ={}
dic_area_of_interest['AIS']= ['']
dic_area_of_interest['WestAntarctica'] = ['_region_1']
dic_area_of_interest['EastAntarctica'] = ['_region_2']
dic_area_of_interest['Peninsula'] = ['_region_3']
dic_area_of_interest['Amundsen'] = ['_sector_4']
dic_area_of_interest['FilchnerRonne'] = ['_sector_5','_sector_11']
dic_area_of_interest['Ross'] = ['_sector_2','_sector_6']

In [19]:
df_dils['WestAntarctica'] = df_dils['region_1']
df_dils['EastAntarctica'] = df_dils['region_2']
df_dils['Peninsula'] = df_dils['region_3']
df_dils_n['WestAntarctica'] = df_dils_n['region_1']
df_dils_n['EastAntarctica'] = df_dils_n['region_2']
df_dils_n['Peninsula'] = df_dils_n['region_3']

In [20]:
if os.path.exists(fpath):
    print( f"The path '{fpath}' already exists.")
else:
    os.makedirs(fpath)
    print( f"The path '{fpath}' did not exist, so it has been created.")
dic_all_err = {}

#for key in ['AIS']:
for key in dic_area_of_interest.keys():
    dic_key = {}
    lst = dic_area_of_interest[key]

    a4_width = 8.27  # inches
    a4_height = 11.69
    colors = ['#1f77b4',
    '#2ca02c',
    '#9467bd',
    '#e377c2']
    fs = 8

    df = df_org[df_org['Experiment']== exp].reset_index(drop=True)
    for j,exp in enumerate(expnames):
        dic_exp ={}
        f,ax = plt.subplots(df.shape[0]//2,2,figsize=(a4_width, a4_height))
        
        df = df_org[df_org['Experiment']== exp].reset_index(drop=True)
        dils_exp =df_dils[df_dils['Experiment']== exp].reset_index(drop=True)
        dils_n_exp =df_dils_n[df_dils_n['Experiment']== exp].reset_index(drop=True)

        for i in range (df.shape[0]):
            ix =i //2
            iy=i%2
            pth_way = df.iloc[i].Path

            #dslc
            pth_dslc = f'{pth_ismip6hack}/{df.iloc[i].Experiment}/{var_dslc}/computed_{var_dslc}_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'
            d3 = xr.open_dataset(pth_dslc,decode_times=False )
        
            #shelf area
            pth_area = f'{pth_helene}/{df.iloc[i].Experiment}/{var_area}/computed_{var_area}_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'
            d2 = xr.open_dataset(pth_area,decode_times=False )
            
            #bmb
            pth_bmb = f'{pth_helene}/{df.iloc[i].Experiment}/{var_bmb}/computed_{var_bmb}_AIS_{df.iloc[i].Model}_{df.iloc[i].Experiment}.nc'
            d4 = xr.open_dataset(pth_bmb,decode_times=False )
            tmin = np.min([d2.time.size,d3.time.size,d4.time.size])

            x = np.cumsum(d4.shelfmelt.values[:tmin]*factor / d2.iareafl.values[:tmin]) #cumulative bmb
            y = np.cumsum(-1*d3.dslc_anom.values[:tmin]) #cumulative dSLC
            dils, dils_n, r_value, p_value, std_err = linregress(x, y)

            ### for some reason the slope / intercepts are not the same when calculated directly above vs using the saved versions???
            # dils = dils_exp.iloc[i][key]
            # dils_n = dils_n_exp.iloc[i][key]
            
            ax2 = ax[ix,iy]            
            ax2.plot(d3.time[:tmin],y ,color = colors[j],label = f'{exp}')
            ax2.plot(d3.time[:tmin],dils*x+dils_n,color = 'k',linestyle ='dashed',label = f'recalc')
            
            err = (dils*x+dils_n - y)**2
            
            dilsk = np.isnan(err) | (y == 0)
            rmse = (np.sum(err[~dilsk])/err[~dilsk].size)**0.5
            prmse = (( (1/err[~dilsk].size)*np.sum(err[~dilsk]))/np.sum(y[~dilsk]**2))**0.5 *100 # %
            dic_exp[df.iloc[i].Model] = prmse.copy()
            
            df_info_model = df_info[df_info['fileID']==df.iloc[i].Model ]
            df_info_model_exp = df_info_model[df_info_model['Experiment']==exp]
            meltparam =  df_info_model_exp['Melt parameterisation'].values[0]
            icefront =  df_info_model_exp['ice front'].values[0]
            
            ax2.text(0.02, 0.98,f' Model {df.iloc[i].Model},\n {meltparam}, {icefront} ',transform=ax2.transAxes,fontsize = fs, va='top',color = 'red')

        dic_key[exp]=dic_exp.copy()
        handles, labels = ax[0, 0].get_legend_handles_labels()
        f.legend(handles, labels, loc='upper right', title='Experiments')
        handles, labels = ax2.get_legend_handles_labels()
        f.legend(handles, labels, loc='upper left', title='Experiments')
        f.suptitle(f'{fname} {key} ice shelf(GT/yr)',fontsize = 12)
        f.savefig(f'{fpath}{fname}{key}_{exp}.pdf')
        plt.close()
    dic_all_err[key] = dic_key.copy()

The path '/Volumes/FMac/computing/data/antarctica/ismip6/hackathon/figures/recalc_dslc_anom_dslcs_dils/' already exists.


In [21]:
df_err = df_dils.copy()
df_err=df_err.drop(df_err.columns[3:21],axis=1)
df_err=df_err.drop(df_err.columns[4:7],axis=1)
for key in dic_area_of_interest.keys():
    df_err[key]=0

for i in range (df_err.shape[0]):
    model =df_err.iloc[i].Model
    exp =df_err.iloc[i].Experiment
    for j,key in enumerate(dic_area_of_interest.keys()):
        df_err.loc[i,key]= dic_all_err[key][exp][model].copy()
df_err.to_csv(pth_table + '/prmse_recalc_linear_dynamic_ice_loss_sensitivity_from_cumBMB.csv')

/var/folders/v3/r3qsjbkx2gl_9r5s_n0726wxcg0fz6/T/ipykernel_42786/2511095448.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.2809464846106617' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_err.loc[i,key]= dic_all_err[key][exp][model].copy()
/var/folders/v3/r3qsjbkx2gl_9r5s_n0726wxcg0fz6/T/ipykernel_42786/2511095448.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.2809464846106617' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_err.loc[i,key]= dic_all_err[key][exp][model].copy()
/var/folders/v3/r3qsjbkx2gl_9r5s_n0726wxcg0fz6/T/ipykernel_42786/2511095448.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.2809464846106617' has dtype incomp